# Pipeline locale de préparation des documents de freinage

**Windows Pro · PDF · Jupyter · sans IA · sans connecteur extérieur**

Ce notebook prépare un corpus pour tes tests : inventaire, texte, tableaux, clauses candidates,
fragments traçables, obligations candidates, recherche et rapport qualité. Il ne juge pas la conformité.

Le moteur est dans `pipeline_local.py`, à côté du notebook. Le séparer du notebook permet une reprise fiable
et des tests automatiques, tout en conservant ici les paramètres et les étapes expliquées.

Commence par **Run → Run All Cells** sur la démonstration fictive de 4 pages. Ensuite, sélectionne tes PDF.
Les packages doivent déjà être installés selon `README.md`. Aucune cellule ne lance pip, un modèle, une API ou un téléchargement.
Les échanges du navigateur avec Jupyter se font uniquement sur ton propre ordinateur (`127.0.0.1`).

| Étape | Entrée | Résultat | Contrôle |
|---|---|---|---|
| 1 | Dossier de PDF | Inventaire et empreintes | Fichiers lisibles, doublons exacts |
| 2 | PDF | Texte brut par page | Texte absent ou mal encodé |
| 3 | Texte brut | Texte nettoyé conservateur | Négations, notes et unités conservées |
| 4 | Géométrie PDF | Tableaux candidats | Cellules vides, tableaux manquants |
| 5 | Texte | Clauses et fragments candidats | Référence de page et offsets |
| 6 | Expressions FR/EN | Blocs d’obligation | Qualification humaine requise |
| 7 | Fragments | Index SQLite local | Recherche exacte et filtres |
| 8 | Résultats | CSV, JSONL, HTML | Échantillonnage et revue métier |

Les tableaux sans bordure, cellules fusionnées et tableaux sur plusieurs pages ne sont pas reconstitués automatiquement.
Les images et dessins sont signalés, sans reconnaissance ni compréhension de leur contenu.


## 1 — Vérifier l’environnement

Utilise de préférence Python 3.11 ou 3.12, 64 bits. Ouvre le notebook depuis le dossier extrait du ZIP.
L’affichage utilise les composants déjà installés par Jupyter. Le traitement n’utilise que pdfplumber et la bibliothèque standard Python.


In [ ]:
from pathlib import Path
import json
import sys
import platform
import importlib.metadata
from IPython.display import display, HTML, Image, FileLink

ROOT = Path.cwd().resolve()
if not (ROOT / "pipeline_local.py").is_file():
    raise RuntimeError("Ouvrir Jupyter depuis le dossier contenant pipeline_local.py et ce notebook.")

from pipeline_local import (
    Config, discover, run_pipeline, search, read_jsonl, html_table,
    render_page, extract_page, extraction_signature,
)

print("Python :", sys.version.split()[0], "| OS :", platform.system())
print("pdfplumber :", importlib.metadata.version("pdfplumber"))
print("Dossier du projet :", ROOT)


## 2 — Choisir le lot et les paramètres

1. Pour le premier essai, garder `MODE_DEMO = True`.
2. Pour tes documents, copier les PDF dans `data/input` puis passer `MODE_DEMO = False`.
3. Tester d’abord 100 pages puis régler `MAX_PAGES = None` pour **toutes les pages** du dossier.
   `MAX_PAGES = 5000` impose une limite globale de 5 000 pages ; les pages suivantes sont explicitement exclues du lot.

Les PDF sont triés par chemin : la limite sélectionne leurs premières pages, pas un échantillon aléatoire.
Pour tester les mises en page difficiles, constituer un dossier contenant des documents représentatifs.

`table_strategy="lines"` utilise les bordures. `"text"` s’appuie sur l’alignement des mots : à essayer
sur un dossier séparé de PDF sans bordures, car cette méthode peut transformer de la prose en faux tableaux.
Le cache est distinct si les paramètres d’extraction changent.


In [ ]:
MODE_DEMO = True
MAX_PAGES = 100  # Après le premier contrôle : None pour le corpus complet.

INPUT_DIR = ROOT / "data" / ("demo" if MODE_DEMO else "input")
OUTPUT_DIR = ROOT / "output" / ("demo" if MODE_DEMO else "corpus")
# Autre emplacement Windows possible, en dehors de output :
# INPUT_DIR = Path(r"C:\Travail\Freinage\PDF")

CONFIG = Config(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    max_pages=MAX_PAGES,
    tables=True,
    table_strategy="lines",
    min_text_chars=60,
    max_chunk_chars=1800,
    max_vector_objects_for_tables=4000,
    progress_every=25,
    retry_errors=True,
)
CONFIG.validate()
print("Entrée :", CONFIG.input_dir)
print("Sortie :", CONFIG.output_dir)
print("Limite :", CONFIG.max_pages if CONFIG.max_pages is not None else "aucune")


## 3 — Examiner l’inventaire avant le traitement

L’empreinte SHA-256 identifie le contenu exact. Deux copies identiques sont dédupliquées ; deux éditions
différentes restent deux documents distincts, même si leur nom ressemble. Un fichier cassé est journalisé
et ne bloque pas les autres PDF. Les PDF protégés qui ne peuvent pas être ouverts sont signalés ; aucun contournement n’est effectué.


In [ ]:
inventory_preview = discover(CONFIG.input_dir)
print("PDF trouvés :", len(inventory_preview))
if not inventory_preview:
    raise ValueError("Déposer les PDF dans le dossier choisi ou conserver MODE_DEMO=True.")
display(HTML(html_table(inventory_preview,
    ["relative_path", "size_bytes", "status", "duplicate_of", "error"], limit=30)))


## 4 — Renseigner les métadonnées documentaires si disponibles

Le notebook ne déduit pas l’édition applicable à partir d’un nom de fichier.
Copier `data/metadata_template.csv` vers `data/metadata.csv` puis remplacer les lignes par tes données.
`relative_path` est le chemin relatif sous le dossier d’entrée, avec le sous-dossier éventuel.
Laisser les champs inconnus vides. Séparateur `;`, encodage UTF-8.

Ces métadonnées documentent les résultats, sans déterminer l’applicabilité réglementaire.


In [ ]:
METADATA_CSV = None
# Pour activer :
# METADATA_CSV = ROOT / "data" / "metadata.csv"
print("Métadonnées :", METADATA_CSV or "non renseignées")


## 5 — Lancer l’extraction et la préparation du corpus

Pour chaque page, le moteur conserve le texte brut extrait et sa version nettoyée. Le nettoyage retire
seulement les espaces superflus et normalise Unicode ; les notes, en-têtes, pieds de page et négations restent présents.
Les tableaux sont enregistrés séparément **et restent dans le texte** : ce choix favorise la traçabilité,
mais crée des répétitions qu’il faudra prendre en compte lors d’une future utilisation.

Chaque page terminée est enregistrée atomiquement dans le cache. Pour interrompre : **Kernel → Interrupt**.
Relancer cette cellule reprend les pages déjà enregistrées. Un changement du contenu d’un PDF ou des paramètres
d’extraction crée un nouveau cache. Un changement de taille des fragments réutilise l’extraction puis refait le découpage.

Une nouvelle exécution produit un nouveau dossier `output/.../runs/...`. Les anciens résultats ne sont pas écrasés.
Un seul traitement peut utiliser le même dossier de sortie à la fois. Après un arrêt brutal du processus,
consulter la section `pipeline.lock` du README.

Le traitement est séquentiel pour commencer ; les données détaillées sont libérées après chaque page.
La durée dépend fortement des tableaux et de la géométrie des PDF. Aucune estimation n’est déduite du nombre de pages seul.


In [ ]:
RUN_DIR = run_pipeline(CONFIG, metadata_csv=METADATA_CSV)
print("Dossier de résultats :", RUN_DIR)


## 6 — Lire les indicateurs de qualité

`pages_selected` = pages du lot exportées, y compris les pages en erreur.
`pages_cached` = pages relues depuis le cache pendant cette exécution.
`tables` = tableaux détectés, pas nombre garanti de tableaux présents.
`obligations_candidates` = blocs contenant un marqueur lexical, pas nombre d’exigences validées.

Un signal « peu de texte » peut correspondre à un scan, mais aussi à une couverture, une page vide ou un schéma.
Les éléments graphiques ne permettent pas de classer sûrement une page.


In [ ]:
summary = json.loads((RUN_DIR / "summary.json").read_text(encoding="utf-8"))
display(HTML(html_table([{
    "pages": summary["pages_selected"],
    "erreurs_pages": summary["pages_error"],
    "erreurs_fichiers": summary["files_error"],
    "tableaux_candidats": summary["tables"],
    "fragments": summary["chunks"],
    "obligations_candidates": summary["obligations_candidates"],
    "pages_cache": summary["pages_cached"],
    "durée_secondes": summary["elapsed_seconds"],
}])))
display(HTML(html_table([{"signal": k, "pages": v} for k,v in summary["flags"].items()])))
display(FileLink((RUN_DIR / "rapport_qualite.html").relative_to(ROOT).as_posix(), result_html_prefix="Rapport HTML local : "))


## 7 — Contrôler le texte extrait et la provenance

Le numéro `page_pdf` commence à 1 et désigne la position dans le fichier, **pas la pagination imprimée**.
`printed_page` reste vide. La source est identifiée par son empreinte et son chemin dans `manifest.json`.
`raw_text` correspond au résultat du parseur : il ne garantit pas l’ordre de lecture d’une mise en page complexe.

Ne charger que quelques lignes JSONL dans le notebook ; le corpus complet peut être volumineux.


In [ ]:
pages_sample = read_jsonl(RUN_DIR / "pages.jsonl", limit=3)
for page in pages_sample:
    print("\nSOURCE :", page["file_name"], "| page PDF", page["page_pdf"], "| édition", page.get("edition") or "inconnue")
    print("Signaux :", page["flags"])
    print(page["clean_text"][:1600])


## 8 — Vérifier les tableaux

`tables.jsonl` contient les cellules et leur rectangle `bbox` en points PDF : gauche, haut, droite, bas.
`tables.csv` donne l’inventaire ; chaque tableau possède aussi un CSV individuel, compatible avec Excel.
Les cellules fusionnées peuvent devenir vides ; les tableaux multipages restent séparés.
Comparer en priorité les entêtes, unités, notes, signes et conditions au PDF.

Pour une protection contre les formules Excel, les cellules CSV commençant par `=`, `+`, `-` ou `@`
sont préfixées par une apostrophe. Les valeurs sans cette protection restent dans JSONL.


In [ ]:
tables_sample = read_jsonl(RUN_DIR / "tables.jsonl", limit=3)
if not tables_sample:
    print("Aucun tableau détecté dans ce lot. Vérifier les pages sources avant toute conclusion.")
for table in tables_sample:
    print("\nTableau :", table["file_name"], "page", table["page_pdf"], "bbox", table["bbox"])
    rows = [{f"col_{i+1}": value for i,value in enumerate(row)} for row in table["cells"][:10]]
    display(HTML(html_table(rows)))
    print("CSV :", RUN_DIR / table["csv_path"])


## 9 — Consulter une page en image

Cette fonction rend une page sur disque local, sans OCR. Elle sert à comparer visuellement le PDF aux données.
Elle est désactivée par défaut pour éviter de produire des images inutiles sur 5 000 pages.
Les graphiques et schémas restent visibles dans ce rendu, sans extraction de leurs relations ou de leurs valeurs.


In [ ]:
AFFICHER_PAGE = False
PAGE_A_VOIR = 2
if AFFICHER_PAGE:
    manifest = json.loads((RUN_DIR / "manifest.json").read_text(encoding="utf-8"))
    selected_doc = next(d for d in manifest["inventory"] if d["status"] in {"processed", "partial"})
    image_path = render_page(Path(selected_doc["path"]), PAGE_A_VOIR,
                             RUN_DIR / "previews" / f"page_{PAGE_A_VOIR}.png")
    display(Image(filename=str(image_path)))
else:
    print("Passer AFFICHER_PAGE à True pour une revue visuelle locale.")


## 10 — Examiner les clauses et les fragments

Des règles repèrent des titres comme `2.1 Freinage` ou `Annexe A`. Leur statut reste « heuristique à confirmer ».
Une clause peut continuer sur la page suivante, mais cette propagation est interrompue après une page illisible ou très pauvre en texte.
Un nombre dans un tableau peut être pris à tort pour un titre. Les sommaires repérés ne créent pas de clauses.

Les fragments ne traversent pas les pages. Ils gardent les offsets `start_char` et `end_char` dans `clean_text`
pour retrouver exactement leur provenance. Le découpage peut couper une obligation longue ; consulter la page et
les fragments voisins. Aucun raisonnement ni enrichissement sémantique n’est effectué.


In [ ]:
chunks_sample = read_jsonl(RUN_DIR / "chunks.jsonl", limit=8)
display(HTML(html_table(chunks_sample,
    ["file_name", "page_pdf", "clause_candidate", "clause_status", "start_char", "end_char", "text"], limit=8)))


## 11 — Revoir les obligations candidates

Le moteur cherche `doit`, `doivent`, `shall`, `must` et quelques variantes. Il conserve la ligne précédente et
deux lignes suivantes pour aider à lire le contexte. Cela peut inclure un titre, un exemple ou une citation.
Une condition située ailleurs, une obligation exprimée sans ces mots ou une obligation répartie entre plusieurs pages
peut être manquée. Les négations ne sont pas supprimées.

L’export propose un champ `commentaire_relecteur`. Les modifications du CSV ne sont pas réimportées automatiquement.
Conserver une copie de revue distincte, car une nouvelle exécution produit un nouvel export.


In [ ]:
obligations_sample = read_jsonl(RUN_DIR / "obligations_candidates.jsonl", limit=12)
display(HTML(html_table(obligations_sample,
    ["file_name", "page_pdf", "clause_candidate", "trigger", "text", "status"], limit=12)))


## 12 — Tester la recherche locale

SQLite FTS5 recherche les mots dans les fragments. Par défaut, tous les mots doivent être présents (`mode="all"`).
`mode="any"` permet une recherche plus large. Les accents sont normalisés par FTS5, mais les synonymes et traductions
ne sont pas inventés : essayer également `braking` pour un corpus anglais.
Si FTS5 n’est pas disponible, le repli `LIKE` est moins performant et sensible aux accents.

Le score est un classement lexical, **pas une confiance de conformité**. Un terme absent du texte extrait
ne signifie pas qu’il est absent des images ou du PDF. Le filtre `doc_id` limite la recherche à un document précis.


In [ ]:
REQUETE = "freinage"
results = search(RUN_DIR, REQUETE, limit=10, mode="all")
display(HTML(html_table(results, ["file_name", "page_pdf", "clause", "edition", "text"], limit=10)))

# Autres essais :
# search(RUN_DIR, "braking emergency", mode="all")
# search(RUN_DIR, "freinage braking", mode="any")
# search(RUN_DIR, "doit", doc_id=pages_sample[0]["doc_id"])


## 13 — Préparer la revue qualité

Pour le premier essai : vérifier environ 20 pages normales, 10 pages avec tableaux et toutes les catégories d’alertes.
Adapter l’échantillon au corpus et à sa criticité. Relever manuellement : texte complet, négations, unités,
association entre titres et contenu, tableaux, renvois et éditions. Une absence d’alertes ne garantit pas l’extraction parfaite.

Avant 5 000 pages, vérifier le résultat des tableaux et le temps par page sur un lot représentatif.
Ensuite passer `MODE_DEMO=False` et `MAX_PAGES=None`, puis relancer à partir de la configuration.
`MAX_PAGES=5000` est possible si tu souhaites plafonner le lot.

Les étapes futures non implémentées sont la recherche sémantique, les embeddings, le RAG, les API,
les connecteurs GED, l’OCR, la compréhension des schémas, l’applicabilité et la décision de conformité.


In [ ]:
for name in ["inventory.csv", "review_pages.csv", "tables.csv", "obligations_candidates.csv",
             "repeated_margins.csv", "rapport_qualite.html", "manifest.json"]:
    display(FileLink((RUN_DIR / name).relative_to(ROOT).as_posix()))
print("Les sorties et caches peuvent contenir le contenu intégral des documents : appliquer les mêmes règles de stockage.")


## 14 — Réouvrir les résultats après un redémarrage

Exécuter la première cellule d’import et la configuration, puis utiliser le code ci-dessous si l’extraction
est déjà terminée. Pour reprendre une extraction interrompue, relancer plutôt l’étape 5.

```python
RUN_DIR = Path(json.loads((CONFIG.output_dir / "latest_run.json").read_text(encoding="utf-8"))["run_dir"])
```

Ne lancer qu’une exécution par dossier `output`. Nettoyer les anciens dossiers uniquement après avoir conservé
les résultats nécessaires ; aucune suppression automatique n’est prévue.
